In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


## setup


In [2]:
!pip install -qU numpy==1.26.4
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1
!pip install -qU faker==35.2.0
!pip install -qU vllm==0.7.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 91.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requi

In [4]:
!pip install -qU locust

In [ ]:
# install LLaMa-Factory
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && pip install -e .

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key = userdata.get('wandb'))
hf_token = userdata.get('huggingface')
!huggingface-cli login --token {hf_token}

## imports

In [3]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel,Field
from typing import List, Optional, Literal
from datetime import datetime
import json_repair
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [4]:
data_dir = "/gdrive/myDrive/projects/llm_finetune_arabic_news"
base_model = 'Qwen/Qwen2.5-1.5B-Instruct'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = None

In [5]:
def parse_json(text):
  try:
    return json_repair.loads(text)
  except:
    return None

# Tasks

In [6]:
story = """
ذكرت مجلة فوربس أن العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال،
 حيث تتأثر هذه العلاقة بأنماط السلوك المالي المتوارثة عبر الأجيال.

التقرير الذي يستند إلى أبحاث الأستاذ الجامعي شاين إنيت حول
الرفاه المالي يوضح أن لكل شخص "شخصية مالية" تتحدد وفقا لطريقة
 تفاعله مع المال، والتي تتأثر بشكل مباشر بتربية الأسرة وتجارب الطفولة.

 الأبعاد الثلاثة للعلاقة بالمال
بحسب الدراسة، هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال:

الاكتساب (A): يميل الأفراد الذين ينتمون لهذا
 البعد إلى اعتبار المال سلعة قابلة للجمع، حيث يرون
في تحقيق الثروة هدفا بحد ذاته. والجانب السلبي لهذا
 النمط هو إمكانية التحول إلى هوس بالثروة أو العكس،
 أي رفض تام لاكتساب المال باعتباره مصدرا للفساد.

الاستخدام (U): يرى هؤلاء الأشخاص المال أداة للتمتع بالحياة، حيث يربطون قيمته بقدرته على توفير
المتعة والراحة. ومع ذلك، قد يصبح
البعض مدمنا على الإنفاق، في حين يتجه آخرون إلى التقشف المفرط خوفا من المستقبل.

الإدارة (M): أصحاب هذا النمط يعتبرون المال مسؤولية تتطلب التخطيط الدقيق. لكن في بعض الحالات،
 قد يتحول الأمر إلى هوس مفرط بإدارة الإنفاق، مما يؤثر سلبا على العلاقات الشخصية.

 كيف تؤثر العائلة على علاقتنا بالمال؟
يشير التقرير إلى أن التجارب الأسرية تلعب دورا رئيسيا في تحديد
 "الشخصية المالية" لكل فرد، على سبيل المثال، إذا كان أحد الوالدين يعتمد على المال
كمكافأة للسلوك الجيد، فقد يتبنى الطفل لاحقا النمط نفسه في حياته البالغة.

لتحليل هذه التأثيرات بشكل دقيق، طورت رابطة العلاج المالي
(Financial Therapy Association) أداة تسمى مخطط الجينوم المالي (Money Genogram)،
وهو نموذج يُستخدم لتحديد الأنماط المالية داخل العائلة.

تتضمن هذه الأداة:

رسم شجرة عائلية.
تصنيف أفراد العائلة وفقا للأبعاد الثلاثة للعلاقة بالمال (A ،U ،M).
تحديد ما إذا كان السلوك المالي لكل فرد صحيا (+) أو غير صحي (-).
على سبيل المثال، إذا نشأ شخص في عائلة
اعتادت على الإنفاق المفرط، فقد يكون لديه ميل قوي إلى اتباع النمط نفسه،
 أو العكس تماما، حيث يصبح مقتصدا بشكل مبالغ فيه كرد فعل نفسي.
"""

# Details Extraction

In [7]:


storyCategory = Literal["politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"]


storyEntities = Literal[  "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"]


class Entity(BaseModel):
  entity_value :str = Field(... , description='the actual name or value of the entity')

  entity_tupe :storyEntities = Field(...,description='the type of recognized entity')


class NewsDetails(BaseModel):
  story_titel: str = Field(...,min_length=10 , max_length=100 ,
                           description='A fully informative and SEO optimized titel of the story.')

  story_keywords: List[str] = Field(..., min_items=2 ,
                                    description = 'Relevant Keywords associated with the story.')

  story_sammary: List[str] = Field(...,min_items = 2 ,max_items = 5,
                                   description='Summarized key points about the story (2-5 points)')

  story_category: storyCategory = Field(...,min_items = 1 ,max_items = 5 ,description='category of the news story.')

  story_entities: List[Entity ] = Field(... ,min_items = 1 ,max_items = 10, description = 'List of identified entities in the story.')


/tmp/ipykernel_13825/4144166596.py:20: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=2 ,
/tmp/ipykernel_13825/4144166596.py:23: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_sammary: List[str] = Field(...,min_items = 2 ,max_items = 5,
/tmp/ipykernel_13825/4144166596.py:23: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_sammary: List[str] = Field(...,min_items = 2 ,max_items = 5,


In [8]:
details_extraction_messages = [
  {
      "role":"system",
      "content":"\n".join([
          "you are an NLP data paraser.",
          "you will be provided by an Arabic text associated with a pydantic schema.",
          "generate the output in the same story language.",
          "you have to extract JSON details from text according the pydantic details.",
          "Extract details as mentiones in text",
          "do not generate any intoduction or conclusion."
      ])
  },
  {

  "role":"user",
  "content":"\n".join([
      "## story",
      story.strip(),"",

      "pydantic details:",
      json.dumps(NewsDetails.model_json_schema() , ensure_ascii=False),
      "",
      "## story Details:",
      "```json"
  ])

     }
]

## translation

In [9]:
class TranslateStory(BaseModel):
  translated_titel : str = Field(..., min_length = 10 , _length = 300 ,
                                 description ="Suggested translated title of the news story.")

  translated_content : str = Field(... , min_length = 10 ,
                                   description = "translated content of the news story.")

/tmp/ipykernel_13825/3776320793.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: '_length'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  translated_titel : str = Field(..., min_length = 10 , _length = 300 ,


In [10]:

targeted_lang = "English"
translation_message = [

            {
                "role":"system",
                "content" : "\n".join([
            "You are a professional translator.",
            "You will be provided by an Arabic text.",
            "You have to translate the text into the `Targeted Language`.",
            "Follow the provided Scheme to generate a JSON",
            "Do not generate any introduction or conclusion."]
                )
            },
            {
                "role":"user",
                "content" : "\n".join([
                    "## story:",story.strip(),"",


                    "## pydantic details:",
                    json.dumps(TranslateStory.model_json_schema(), ensure_ascii = False),
                    "",

                    "## target language:",targeted_lang,

                    "## translated story:",
                    "```json"
                ])
            }]

In [ ]:
NewsDetails.model_json_schema()

# Evaluation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model, # model
    device_map = "auto", # gpu
    torch_dtype = torch_dtype, #load the llm on gpu and reduce its size it make it fit on gpu and faster but it have side effecto on quality

)

tokenizer = AutoTokenizer.from_pretrained(base_model)


In [ ]:
text = tokenizer.apply_chat_template(
    details_extraction_messages, # the form of the maessage we have created
    tokenize = False, #show the message as words not tokens
    add_generation_prompet = True # add geenration prompet to make the model rady to generate
)

model_inputs = tokenizer([text] , return_tensors='pt').to(device)

generate_id = model.generate(
    model_inputs.input_ids,
    max_new_tokens = 1024,
    do_sample = True,
    top_k = None,
    temperature = None,
    top_p = None,
)

generate_id = [
    output_ids[len(input_ids):] for input_ids , output_ids in zip(model_inputs.input_ids,generate_id)
]


response = tokenizer.batch_decode(generate_id , skip_special_tokens=True)

In [ ]:
response # details extraction

In [ ]:
text = tokenizer.apply_chat_template(
    translation_message, # the form of the maessage we have created
    tokenize = False, #show the message as words not tokens
    add_generation_prompet = True # add geenration prompet to make the model rady to generate
)

model_inputs = tokenizer([text] , return_tensors='pt').to(device)

generate_id = model.generate(
    model_inputs.input_ids,
    max_new_tokens = 1024,
    do_sample = True,
    top_k = None,
    temperature = None,
    top_p = None,
)

generate_id = [
    output_ids[len(input_ids):] for input_ids , output_ids in zip(model_inputs.input_ids,generate_id)
]


response = tokenizer.batch_decode(generate_id , skip_special_tokens=True)

In [ ]:
response # translate

## Evaluation Deepseek huggingface

In [ ]:
from openai import OpenAI
deepseek_model = 'deepseek-ai/DeepSeek-R1:novita'

In [ ]:
deepseek = OpenAI(
    base_url = "https://router.huggingface.co/v1",
    api_key = userdata.get('deepseek_r1'),

)



In [ ]:
chat_completion = deepseek.chat.completions.create(
    messages = details_extraction_messages,
    model = deepseek_model,
    temperature = 0.2,
    )

In [ ]:
print(chat_completion.choices[0].message.content)

In [ ]:
chat_completion = deepseek.chat.completions.create(
    messages = translation_message,
    model = deepseek_model,
    temperature = 0.2,
    )

print(chat_completion.choices[0].message.content)

## Evaluation OpenAi

## Knowledge Distillation

In [ ]:
raw_data_path = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/news-sample.jsonl"

raw_data = []
for line in open(raw_data_path) :
  if line.strip() == "":
    continue

  raw_data.append(json.loads(line))


random.Random(42).shuffle(raw_data)


## Format Finetuning Datasets

#### configs

In [ ]:
from openai import OpenAI
deepseek_model = 'deepseek-ai/DeepSeek-R1:novita'

deepseek = OpenAI(
    base_url = "https://router.huggingface.co/v1",
    api_key = userdata.get('deepseek_r1'),

)

In [ ]:
def parse_json(text):
  try:
    return json_repair.loads(text)
  except:
    return None

In [ ]:
prompt_tokens = 0
completion_tokens = 0

save_to = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft2.jsonl"

ix= 0

for story in tqdm(raw_data):

  sample_details_extraction_messages = [
    {
        "role":"system",
        "content":"\n".join([
            "you are an NLP data paraser.",
            "you will be provided by an Arabic text associated with a pydantic schema.",
            "generate the output in the same story language.",
            "you have to extract JSON details from text according the pydantic details.",
            "Extract details as mentiones in text",
            "do not generate any intoduction or conclusion."
        ])
    },
    {

    "role":"user",
    "content":"\n".join([
        "## story",
        story['content'].strip(),"",

        "pydantic details:",
        json.dumps(NewsDetails.model_json_schema() , ensure_ascii=False),
        "",
        "## story Details:",
        "```json"
    ])

      }]


  response = deepseek.chat.completions.create(
      messages = sample_details_extraction_messages,
      model = deepseek_model,
      temperature = 0.2,
      )

  if response.choices[0].finish_reason != "stop":
    continue

  llm_response = response.choices[0].message.content
  llm_response = parse_json(llm_response)

  if llm_response is None:
    continue

  with open(save_to , "a" , encoding = "utf-8") as dest:
    dest.write(json.dumps({
        "id":ix,
        "story":story['content'].strip(),
        "task":"Extrat the story details into a JSON.",
        "output_schema":json.dumps(NewsDetails.model_json_schema() , ensure_ascii=False),
        "response":llm_response,

    },ensure_ascii=False , default =str) + "\n")

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens


    print(f"iteration number {ix} number of prompt tokens {prompt_tokens} number of completion tokens {completion_tokens}")


    if(ix % 5) == 0:
      break

In [ ]:
with open(save_to , 'r' ,encoding='utf-8') as src:
  for line in src:
    print(line)
    break

In [ ]:
prompt_tokens = 0
completion_tokens = 0

save_to_2 = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft3.jsonl"

ix= 0
targeted_lang = "English"
for story in tqdm(raw_data):

  sample_translation_message = [

              {
                  "role":"system",
                  "content" : "\n".join([
              "You are a professional translator.",
              "You will be provided by an Arabic text.",
              "You have to translate the text into the `Targeted Language`.",
              "Follow the provided Scheme to generate a JSON",
              "Do not generate any introduction or conclusion."]
                  )
              },
              {
                  "role":"user",
                  "content" : "\n".join([
                      "## story:",story["content"].strip(),"",


                      "## pydantic details:",
                      json.dumps(TranslateStory.model_json_schema(), ensure_ascii = False),
                      "",

                      "## target language:",targeted_lang,

                      "## translated story:",
                      "```json"
                  ])
              }]


  response = deepseek.chat.completions.create(
      messages = sample_translation_message,
      model = deepseek_model,
      temperature = 0.2,
      )

  if response.choices[0].finish_reason != "stop":
    continue

  llm_response = response.choices[0].message.content
  llm_response = parse_json(llm_response)

  if llm_response is None:
    continue

  with open(save_to_2 , "a" , encoding = "utf-8") as dest:
    dest.write(json.dumps({
        "id":ix,
        "story":story['content'].strip(),
        "task":f"You have to translate the story content into {targeted_lang} associated with a title into a JSON.",
        "output_schema":json.dumps(TranslateStory.model_json_schema() , ensure_ascii=False),
        "response":llm_response,

    },ensure_ascii=False , default =str) + "\n")

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens


    print(f"iteration number {ix} number of prompt tokens {prompt_tokens} number of completion tokens {completion_tokens}")


    if(ix % 2) == 0:
      break

In [ ]:
with open(save_to_2 , 'r' ,encoding='utf-8') as src:
  for line in src:
    print(line)
    break

## Format Finetuning Datasets

### new sammary

In [ ]:
sft_data_path = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft.jsonl"

In [ ]:

llm_finetunning_data = []

system_message = "\n".join([
     "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
  if line.strip() == '':
    continue

  rec = json.loads(line.strip())

  llm_finetunning_data.append({
      "system":system_message,
      "instruction" :"\n".join([
      "# Story:",rec['story'],

      "# Task:" , rec['task'],

      "# Output Scheme:",rec['output_scheme'],"",

      "# Output JSON:",
      "```json",
      ]),
      "input":"",
      "output":"\n".join([
          "```json",
     json.dumps(rec['response'], ensure_ascii=False),
      "```"
      ]) ,
        "history": []
  })

In [ ]:
random.Random(42).shuffle(llm_finetunning_data)

In [ ]:
len(llm_finetunning_data)

In [ ]:
data_dir = r"/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets"

train_dample_sz = 2600

train_ds = llm_finetunning_data[:train_dample_sz]
eval_ds = llm_finetunning_data[train_dample_sz:]

os.makedirs(join(data_dir,"llamafactory-finetune-data") , exist_ok = True)

with open(join(data_dir,"llamafactory-finetune-data","train.json"),'w') as dest:
  json.dump(train_ds , dest , ensure_ascii=False , default=str)

with open(join(data_dir,"llamafactory-finetune-data","val.json"),'w') as dest:
  json.dump(eval_ds , dest , ensure_ascii=False , default=str)

### translation

In [ ]:
xsft_data_path = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/xsft.jsonl"

In [ ]:

llm_finetunning_data_t = []

system_message = "\n".join([
     "You are a professional Translator.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
  if line.strip() == '':
    continue

  rec = json.loads(line.strip())

  llm_finetunning_data_t.append({
      "system":system_message,
      "instruction" :"\n".join([
      "# Story:",rec['story'],

      "# Task:" , rec['task'],

      "# Output Scheme:",rec['output_scheme'],"",

      "# Output JSON:",
      "```json",
      ]),
      "input":"",
      "output":"\n".join([
          "```json",
     json.dumps(rec['response'], ensure_ascii=False),
      "```"
      ]) ,
        "history": []
  })

In [ ]:
len(llm_finetunning_data_t)

In [ ]:
data_dir = r"/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets"

train_dample_sz = 2600

train_ds = llm_finetunning_data_t[:train_dample_sz]
eval_ds = llm_finetunning_data_t[train_dample_sz:]

os.makedirs(join(data_dir,"translate_llamafactory-finetune-data") , exist_ok = True)

with open(join(data_dir,"translate_llamafactory-finetune-data","train.json"),'w') as dest:
  json.dump(train_ds , dest , ensure_ascii=False , default=str)

with open(join(data_dir,"translate_llamafactory-finetune-data","val.json"),'w') as dest:
  json.dump(eval_ds , dest , ensure_ascii=False , default=str)

## Finetuning

In [ ]:
# # Configure LLaMA-Factory for the new datasets

# # update /content/LLaMA-Factory/data/dataset_info.json and append
# ```

  #  "news_finetune_train": {
  #       "file_name": "/gdrive/MyDrive/projects/llm_finetune_arabic_news/dataset/llamafactory-finetune-data/train.json",
  #       "columns": {
  #           "prompt": "instruction",
  #           "query": "input",
  #           "response": "output",
  #           "system": "system",
  #           "history": "history"
  #       }
  #   },
  #   "news_finetune_val": {
  #       "file_name": "/gdrive/MyDrive/projects/llm_finetune_arabic_news/dataset/llamafactory-finetune-data/val.json",
  #       "columns": {
  #           "prompt": "instruction",
  #           "query": "input",
  #           "response": "output",
  #           "system": "system",
  #           "history": "history"
  #       }
  #   }

# ```

# https://wandb.ai/mr-bakrianoo/llamafactory/runs/apwbkni9
# https://wandb.ai/mr-bakrianoo/llamafactory/runs/c5tf0q90

In [ ]:
%%writefile /content/LlamaFactory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16
dataloader_num_workers: 4

### output
# resume_from_checkpoint: null
output_dir: /gdrive/MyDrive/projects/llm_finetune_arabic_news/models
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true
# save_only_model: false


### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000


### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 500

report_to: wandb  # choices: [none, wandb, tensorboard, swanlab, mlflow]
run_name: news-finetune-llamafactory

# push_to_hub: true
# export_hub_model_id: abouomar5/news-finetune-llamafactory
# hup_private_repo: true
# hup_strategy: checkpoint

In [ ]:
!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/news_finetune.yaml

## New Finetuned Model Evaluation

In [11]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
finetuned_model_id = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
model.load_adapter(finetuned_model_id)

In [13]:
from peft import PeftModel

finetuned_model_id = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
model = PeftModel.from_pretrained(
    model,
    finetuned_model_id,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


##### translation

In [14]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(translation_message)

# details_extraction_messages
# translation_message

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [15]:
parse_json(response)

{'translated_titel': 'The Role of Family in Financial Relationships',
 'translated_content': 'Forbes magazine reported that the family plays a pivotal role in shaping individuals\' relationship with money, as this relationship is influenced by inherited financial behaviors across generations.\n\nThe report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\nThe three dimensions of our financial relationship according to the study include:\n\nAcquisition (A): Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside of this pattern is the potential for it to turn into an obsession with wealth or vice versa, meaning complete rejection of acquiring money as a means of corruption.\n\nUsage (U)

##### details extraction


In [24]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(details_extraction_messages)

# details_extraction_messages
# translation_message

In [25]:
parse_json(response)

{'story_titel': 'تأثير العائلة على علاقة الأفراد بالمال',
 'story_keywords': ['العائلة',
  'ال金钱',
  'السلوكيات المالية',
  'الشخصية المالية',
  'التاريخ الشخصي'],
 'story_sammary': ['العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال.',
  'الشخصية المالية لكل شخص تعتمد على أنماط السلوك المالي المتوارثة.',
  'الثلاثة أبعاد رئيسية تشمل الاكتساب، الاستخدام، والإدارة.',
  "تجارب الأسرة تؤثر على تحديد 'الشخصية المالية' لكل فرد.",
  'تخطيط دقيق للمال يمكن أن يؤدي إلى هوس مفرط.'],
 'story_category': 'economy',
 'story_entities': [{'entity_value': 'فوربس', 'entity_tupe': 'organization'},
  {'entity_value': 'شاين إنيت', 'entity_tupe': 'person-male'},
  {'entity_value': 'رابطة العلاج المالي', 'entity_tupe': 'organization'},
  {'entity_value': 'Money Genogram', 'entity_tupe': 'artifact'}]}

#### Tip for Qwen2.5

Qwen2.5 oftenly produce chinese characters with some responses. To skip this, use the next class to generate responses.

Source:
`https://jupyter267.medium.com/how-to-eliminate-the-chance-of-generating-chinese-in-qwen-2-5-2cf919bb0fdc`

In [31]:
class Generator:
    def __init__(self, model, tokenizer):

        self.model, self.tokenizer = model, tokenizer
        self.mask = None

    def generate(self, messages:list, max_new_tokens: int=2000, temperature:float=0.1):

        def logits_processor(token_ids, logits):
          # logits_processor default recieve the logits which is the score matrix of each time-step
          """
              A processor to ban Chinese character
          """
          if self.mask is None:
              # as we don't know where the Chinses tokens locate at which index
              # in the vocabulary but we know how it looks like and the range of it

              # decode all the tokens in the vocabulary in order
              token_ids = torch.arange(logits.size(-1))
              decoded_tokens = self.tokenizer.batch_decode(token_ids.unsqueeze(1), skip_special_tokens=True)

              # create a mask tensor to exclude positions of Chinese characters.
              # since this process uses a for loop and is time-consuming,
              # the result will be stored as a property for later use to ensure it only runs once.
              self.mask = torch.tensor([
                  # loop through each token in the vocabulary and compare it to Chinese characters.
                  any(0x4E00 <= ord(c) <= 0x9FFF or 0x3400 <= ord(c) <= 0x4DBF or 0xF900 <= ord(c) <= 0xFAFF for c in
                      token)
                  for token in decoded_tokens
              ])

          # mask the score by - inf
          logits[:, self.mask] = -float("inf")
          return logits

        # this step transforms the messages into a string,
        # adding special tokens e.g separate tokens between system content user queries
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            # add the logits_processor here
            logits_processor=[logits_processor]
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

In [38]:
# define an object
llm = Generator(model, tokenizer)

In [39]:
# generate a response without chinese characters
response = llm.generate(details_extraction_messages)
parse_json(response)

{'story_titel': 'تأثير العائلة على علاقة الأفراد بالمال',
 'story_keywords': ['العائلة',
  'الملعب',
  'الشخصية المالية',
  'التاريخ المالي',
  'الإرشاد المالي'],
 'story_sammary': ['العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال.',
  'الشخصية المالية لكل شخص تعتمد على أنماط السلوك المالي المتوارثة.',
  'الثلاثة أبعاد رئيسية تشمل الاكتساب، الاستخدام، والإدارة.',
  "تجارب الأسرة تؤثر على تحديد 'الشخصية المالية' لكل فرد.",
  'تخطيط دقيق للمال يمكن أن يؤدي إلى هوس مفرط.'],
 'story_category': 'الاقتصاد',
 'story_entities': [{'entity_value': 'فوربس', 'entity_tupe': 'organization'},
  {'entity_value': 'شاين إنيت', 'entity_tupe': 'person-male'},
  {'entity_value': 'رابطة العلاج المالي', 'entity_tupe': 'organization'},
  {'entity_value': 'Money Genogram', 'entity_tupe': 'artifact'}]}

In [40]:
response = llm.generate(translation_message)
parse_json(response)

{'translated_titel': 'The Role of Family in Financial Relationships',
 'translated_content': 'Forbes magazine reported that the family plays a pivotal role in shaping individuals\' relationship with money, as this relationship is influenced by inherited financial behaviors across generations.\n\nThe report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\nThe three dimensions of our financial relationship according to the study include:\n\nAcquisition (A): Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside of this pattern is the potential for it to turn into an obsession with wealth or vice versa, meaning complete rejection of acquiring money as a means of corruption.\n\nUsage (U)

## Cost Estimation

In [26]:
from tqdm.auto import tqdm
from faker import Faker
import random
from datetime import datetime


start_time = datetime.now()
fake = Faker('ar')

input_tokens = 0
output_tokens = 0

for i in tqdm( range(30)):
  prompt = fake.text(max_nb_chars=random.randint(150,300))

  messages = [
      {
          "role":"user",
          "content":prompt
      }
  ]

  response = generate_resp(messages)


  input_tokens += len(tokenizer.apply_chat_template(messages))
  output_tokens += len(tokenizer.encode(response))

  total_time = (datetime.now() - start_time).total_seconds()



print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

  0%|          | 0/30 [00:00<?, ?it/s]

Total Time: 678.602224 seconds
Input Tokens: 2822
Output Tokens: 13364
Total Tokens: 16186


In [29]:

# Total Time: 678.602224 seconds
# Input Tokens: 2822
# Output Tokens: 13364
# Total Tokens: 16186

In [27]:
16186/678

23.87315634218289

## vLLM

In [13]:

base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_model_id = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"


!nohup vllm serve "{base_model_id}" --dtype=half --gpu-memory-utilization 0.8 --max_lora_rank 64 --enable-lora --lora-modules news-lora="{adapter_model_id}" &


nohup: appending output to 'nohup.out'


In [18]:
!tail -n 30 nohup.out

INFO 03-14 23:15:56 model_runner.py:1562] Graph capturing finished in 45 secs, took 0.26 GiB
INFO 03-14 23:15:56 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 65.63 seconds
INFO 03-14 23:15:56 api_server.py:756] Using supplied chat template:
INFO 03-14 23:15:56 api_server.py:756] None
INFO 03-14 23:16:09 serving_models.py:174] Loaded new LoRA adapter: name 'news-lora', path '/gdrive/MyDrive/projects/llm_finetune_arabic_news/models'
INFO 03-14 23:16:09 launcher.py:21] Available routes are:
INFO 03-14 23:16:09 launcher.py:29] Route: /openapi.json, Methods: GET, HEAD
INFO 03-14 23:16:09 launcher.py:29] Route: /docs, Methods: GET, HEAD
INFO 03-14 23:16:09 launcher.py:29] Route: /docs/oauth2-redirect, Methods: GET, HEAD
INFO 03-14 23:16:09 launcher.py:29] Route: /redoc, Methods: GET, HEAD
INFO 03-14 23:16:09 launcher.py:29] Route: /health, Methods: GET
INFO 03-14 23:16:09 launcher.py:29] Route: /ping, Methods: GET, POST
INFO 03-14 23:16:09 launcher.py:29] Rout

## inference

In [19]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

prompt = tokenizer.apply_chat_template(
    translation_message, # the form of the maessage we have created
    tokenize = False, #show the message as words not tokens
    add_generation_prompt  = True # add geenration prompet to make the model rady to generate
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [20]:
vllm_model_id = "news-lora"

llm_response = requests.post("http://localhost:8000/v1/completions", json={
    "model": vllm_model_id,
    "prompt": prompt,
    "max_tokens": 1000,
    "temperature": 0.3
})

llm_response.json()

{'id': 'cmpl-2b7ab4f8489c4f66873f42772132699c',
 'object': 'text_completion',
 'created': 1773530211,
 'model': 'news-lora',
 'choices': [{'index': 0,
   'text': '```json{"translated_titel": "The Role of Family in Financial Relationships", "translated_content": "Forbes magazine reported that the family plays a pivotal role in shaping an individual\'s relationship with money, as this relationship is influenced by inherited financial behaviors across generations. The report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' that is determined by how they interact with money, which is directly affected by family upbringing and childhood experiences. The three dimensions of the financial relationship according to the study include:\\n\\nA: Accumulation: Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside o

## Load Testing

In [22]:
%%writefile locust.py

import random
import json
from locust import HttpUser, task, between, constant
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "news-lora"
        prompt = fake.text(max_nb_chars=random.randint(150, 200))

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.3
        }

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")


Writing locust.py


In [23]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

[2026-03-14 23:18:47,633] 29f78f4fd75e/INFO/locust.main: Starting Locust 2.43.3
[2026-03-14 23:18:47,644] 29f78f4fd75e/INFO/locust.main: Run time limit set to 60 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-03-14 23:18:47,658] 29f78f4fd75e/INFO/locust.runners: Ramping to 20 users at a rate of 1.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

Type     Na

In [24]:
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

In [25]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")

Total Input Tokens: 1336
Total Output Tokens: 18664


In [26]:
18664/60

311.06666666666666

In [3]:
import subprocess, time

proc = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype=half",
    "--gpu-memory-utilization", "0.85",
    "--max_lora_rank", "64",
    "--enable-lora",
    "--lora-modules", "news-lora=/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
])

print("Waiting for vLLM to start...")
time.sleep(90)
print("vLLM should be running on http://localhost:8000")

Waiting for vLLM to start...
vLLM should be running on http://localhost:8000


In [4]:
import subprocess, time

proc = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype=half",
    "--gpu-memory-utilization", "0.85",
    "--max_lora_rank", "64",
    "--enable-lora",
    "--lora-modules", "news-lora=/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
],
stderr=subprocess.PIPE,
stdout=subprocess.PIPE
)

print("Waiting for vLLM...")
for i in range(50):
    line = proc.stdout.readline()
    if line:
        print(line.decode('utf-8', errors='ignore').strip())

Waiting for vLLM...
INFO 03-17 22:08:38 __init__.py:190] Automatically detected platform cuda.
INFO 03-17 22:08:42 api_server.py:840] vLLM API server version 0.7.2
INFO 03-17 22:08:42 api_server.py:841] args: Namespace(subparser='serve', model_tag='Qwen/Qwen2.5-1.5B-Instruct', config='', host=None, port=8000, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=[LoRAModulePath(name='news-lora', path='/gdrive/MyDrive/projects/llm_finetune_arabic_news/models', base_model_name=None)], prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, enable_reasoning=False, reasoning_parser=None, tool_call_parser=None, t

In [5]:
import requests, time


for i in range(24):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print("vLLM ready")
            break
    except:
        print(f"still working {(i+1)*5}s")
        time.sleep(5)

vLLM ready


In [6]:
import requests
r = requests.get("http://localhost:8000/health")
print(r.status_code)


200


In [8]:
!pip install pyngrok

In [9]:
from pyngrok import ngrok

ngrok.set_auth_token("")

url = ngrok.connect(8000)
print("vLLM URL:", url)

vLLM URL: NgrokTunnel: "https://autotomic-buckishly-adele.ngrok-free.dev" -> "http://localhost:8000"
